# Experiment 008: SmolLM2-360M — Full Fine-Tune on 100K Routed Data

**Model:** HuggingFaceTB/SmolLM2-360M-Instruct (362M params, all trainable)

**Method:** Full fine-tune (NOT LoRA — proven insufficient for grammar learning)

**Data:** ~95K train / ~5K eval — balanced 35% ADD, 35% SUB, 15% BETWEEN, 15% NO_ROUTE

**Steps:** 10,000 (~1.6 epochs)

**Objective:** Achieve >60% E2E accuracy (up from 25% in Exp 004) by fixing:
1. ADD/SUB directional confusion (balanced data)
2. Duration hallucination (100× more variety)
3. Memorization (1.6 epochs vs 7 epochs)

---

⚙️ **Recommended runtime:** Runtime → Change runtime type → **L4 GPU**

L4 is the sweet spot for a 360M model in float32 (~24GB VRAM, ~30 TFLOPS). T4 works but is 3-4× slower. A100/H100 are overkill for this model size.

## 1. Setup & Dependencies

In [ ]:
!pip install -q transformers datasets accelerate torch
!pip install -q tensorboard

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Upload Training Data

Upload `train_routed_smollm2.jsonl` and `eval_routed_smollm2.jsonl` from `data/train/` and `data/eval/`.

Or use the cell below to pull from GitHub:

In [ ]:
# Option A: Upload files manually via Colab sidebar
# Option B: Clone repo
import os
if not os.path.exists('tagzeit-gemma-2b'):
    !git clone https://github.com/mgosal/tagzeit-gemma-2b.git
    print("✅ Repo cloned")
else:
    !cd tagzeit-gemma-2b && git pull
    print("✅ Repo updated")

TRAIN_FILE = 'tagzeit-gemma-2b/data/train/train_routed_smollm2.jsonl'
EVAL_FILE = 'tagzeit-gemma-2b/data/eval/eval_routed_smollm2.jsonl'

# Verify
import subprocess
train_lines = int(subprocess.check_output(['wc', '-l', TRAIN_FILE]).split()[0])
eval_lines = int(subprocess.check_output(['wc', '-l', EVAL_FILE]).split()[0])
print(f"Train: {train_lines} samples")
print(f"Eval:  {eval_lines} samples")

## 3. Load Model & Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,  # Full precision for full fine-tune stability
    device_map="auto",
)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Vocab size: {tokenizer.vocab_size}")

## 4. Add Domain Tokens

Register the 104 routing tokens and initialise embeddings with geometric sinusoidal scheme.

In [ ]:
import math
import numpy as np

# Define domain tokens
ROUTE_TOKENS = [
    # Structural
    "[ROUTE]", "[/ROUTE]", "[NO_ROUTE]",
    # Operations
    "[ROUTE_TIME_ADD]", "[ROUTE_TIME_SUB]", "[ROUTE_DURATION_BETWEEN]",
    "[ROUTE_CALENDAR_SHIFT]", "[ROUTE_TIMEZONE_CONVERT]",
    # Heads
    "[HEAD_TIME]", "[HEAD_DURATION]", "[HEAD_DATE]",
    # References
    "[REF_1]", "[REF_2]", "[REF_3]",
]

# Hour tokens: [ARG_HOUR_00] to [ARG_HOUR_23]
ROUTE_TOKENS += [f"[ARG_HOUR_{str(h).zfill(2)}]" for h in range(24)]

# Minute tokens: [ARG_MIN_00] to [ARG_MIN_59]
ROUTE_TOKENS += [f"[ARG_MIN_{str(m).zfill(2)}]" for m in range(60)]

# Day-of-week tokens
ROUTE_TOKENS += ["[ARG_MON]", "[ARG_TUE]", "[ARG_WED]", "[ARG_THU]", "[ARG_FRI]", "[ARG_SAT]", "[ARG_SUN]"]

print(f"Domain tokens to add: {len(ROUTE_TOKENS)}")

# Add tokens
num_added = tokenizer.add_tokens(ROUTE_TOKENS)
model.resize_token_embeddings(len(tokenizer))
print(f"Added {num_added} tokens. New vocab size: {len(tokenizer)}")

# Geometric sinusoidal initialisation
embed_dim = model.config.hidden_size
with torch.no_grad():
    new_start = len(tokenizer) - num_added
    for i in range(num_added):
        token_id = new_start + i
        # Geometric sinusoidal: each token gets a unique frequency pattern
        freq = np.array([1.0 / (10000 ** (2 * j / embed_dim)) for j in range(embed_dim)])
        phase = (i + 1) * freq
        init_vec = np.sin(phase) * 0.02  # Scale down to match embedding magnitude
        model.get_input_embeddings().weight[token_id] = torch.tensor(init_vec, dtype=torch.float32)

print(f"✅ Embeddings initialised (geometric sinusoidal)")

## 5. Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train': TRAIN_FILE,
    'eval': EVAL_FILE,
})

print(f"Train: {len(dataset['train'])} samples")
print(f"Eval:  {len(dataset['eval'])} samples")
print(f"\nSample:")
print(dataset['train'][0]['text'][:300])

In [ ]:
# Tokenize
MAX_SEQ_LEN = 256  # Route format is compact

def tokenize_fn(examples):
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding='max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

print(f"Tokenized train: {len(tokenized['train'])}")
print(f"Tokenized eval:  {len(tokenized['eval'])}")

# Verify token counts
sample_len = (tokenized['train'][0]['input_ids'] != tokenizer.pad_token_id).sum().item()
print(f"Sample sequence length (non-pad): {sample_len}")

## 6. Training Configuration

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "008-smollm2-360m-fullft-100k"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Steps
    max_steps=10000,
    
    # Batch size: 4 per device × 4 gradient accumulation = 16 effective
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    
    # Optimizer
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=200,
    weight_decay=0.01,
    optim="adamw_torch",
    
    # Precision — full float32 for stability
    bf16=True,  # L4/A100 support bf16 natively — 4x faster than float32
    fp16=False,
    
    # Evaluation & Checkpointing
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=10,
    
    # Logging
    logging_steps=50,
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="tensorboard",
    
    # Other
    dataloader_num_workers=2,
    seed=42,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# Calculate training info
effective_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
samples_per_step = effective_batch
total_samples_seen = training_args.max_steps * samples_per_step
epochs_approx = total_samples_seen / len(tokenized['train'])

print(f"Effective batch size: {effective_batch}")
print(f"Total samples seen: {total_samples_seen:,}")
print(f"Approximate epochs: {epochs_approx:.1f}")
print(f"Checkpoints: every {training_args.save_steps} steps")

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['eval'],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("✅ Trainer ready")

## 7. Train!

In [ ]:
train_result = trainer.train()

print(f"\n{'='*50}")
print(f"Training complete!")
print(f"  Train loss: {train_result.training_loss:.4f}")
print(f"  Steps: {train_result.global_step}")
print(f"  Runtime: {train_result.metrics['train_runtime']:.0f}s")
print(f"  Samples/sec: {train_result.metrics['train_samples_per_second']:.1f}")

## 8. Evaluate

In [ ]:
eval_results = trainer.evaluate()
print(f"\nEval loss: {eval_results['eval_loss']:.4f}")
print(f"Eval perplexity: {math.exp(eval_results['eval_loss']):.2f}")

## 9. Quick Inference Test

In [ ]:
test_prompts = [
    "The meeting starts at 14:30 and lasts 45 minutes. When does it end?",
    "I arrived at work at 09:15. The commute was 25 minutes. When did I leave home?",
    "The movie ran from 19:00 to 21:30. How long was it?",
    "What is 42 + 18?",
    "I put the cake in at half past two. It bakes for an hour and a half. When should I take it out?",
]

model.eval()
for prompt in test_prompts:
    messages = [
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=1.0,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    # Clean up
    response = response.split('<|im_end|>')[0].strip()
    
    print(f"Q: {prompt}")
    print(f"A: {response}")
    print()

## 10. Save Final Model

In [ ]:
# Save best model
trainer.save_model(f"{OUTPUT_DIR}/best")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best")
print(f"✅ Best model saved to {OUTPUT_DIR}/best")

# List checkpoints
import glob
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
print(f"\nCheckpoints saved:")
for cp in checkpoints:
    print(f"  {cp}")

## 11. Download Checkpoints

Zip up checkpoints for local evaluation with `validate_route.py`.

In [ ]:
import shutil

# Zip the best model and key checkpoints for download
for checkpoint_dir in [f"{OUTPUT_DIR}/best"] + checkpoints:
    name = checkpoint_dir.replace('/', '_')
    shutil.make_archive(name, 'zip', checkpoint_dir)
    print(f"📦 {name}.zip")

print("\n✅ Download these zip files from the Colab file browser.")

## 12. TensorBoard

View loss curves and training metrics.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}/logs